In [54]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("File Processing") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [78]:
from google.colab import files
uploaded = files.upload()

Saving sales.csv to sales.csv
Saving employees.csv to employees (2).csv
Saving logins.txt to logins (3).txt


In [56]:
!ls

'employees (1).csv'  'logins (1).txt'   logins.txt
 employees.csv	     'logins (2).txt'   sample_data


Questions (TXT Processing)
1. Read the TXT file and print all names.


In [57]:
with open('logins.txt', 'r') as file:
  lines = file.readlines()
  for line in lines:
    print(line)

Rahul

Sneha

Arjun

Rahul

Priya

Sneha

Rahul

Karan

Arjun

Sneha

Rahul

Amit

Priya

Karan

Sneha

Rahul

Meera

Arjun

Sneha

Rahul

Karan

Amit

Priya

Sneha

Rahul

Arjun


2. Count the total login events.


In [58]:
len(lines)

26

3. Find the unique users.


In [59]:
lines = [i.strip() for i in lines]
set(lines)

{'Amit', 'Arjun', 'Karan', 'Meera', 'Priya', 'Rahul', 'Sneha'}

4. Count how many times each user logged in.

In [60]:
for i in set(lines):
  print(i, lines.count(i))

Amit 2
Rahul 7
Karan 3
Priya 3
Sneha 6
Arjun 4
Meera 1


5. Find the top 3 most active users.


In [61]:
from collections import Counter

user_counts = dict(Counter(lines))

top_3 = sorted(user_counts.items(), key=lambda x: x[1], reverse=True)[:3]
print("Top 3 Users:", top_3)

Top 3 Users: [('Rahul', 7), ('Sneha', 6), ('Arjun', 4)]


6. Find users who logged in more than 4 times.

In [62]:
for i in user_counts.items():
  if i[1] > 4:
    print(i)

('Rahul', 7)
('Sneha', 6)


7. Convert the login list into a dictionary of counts.

Expected format example:

{
"Rahul": 7,
"Sneha": 6,
"Arjun": 4
}

In [63]:
print(user_counts)

{'Rahul': 7, 'Sneha': 6, 'Arjun': 4, 'Priya': 3, 'Karan': 3, 'Amit': 2, 'Meera': 1}


Dataset 2 — CSV (Employees Dataset)
File: employees.csv

In [64]:
from pyspark.sql.functions import col
import csv

Questions (CSV Processing)
1. Load the CSV file.


In [65]:
employees = spark.read.csv('employees.csv', header=True, inferSchema=True)
employees.show(5)

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
+------+-----+----------+------+---------+
only showing top 5 rows


2. Count total employees.


In [66]:
emp_count = employees.count()
print("Total Employees:", emp_count)

Total Employees: 20


3. Show employees from IT department.

In [67]:
with open("employees.csv", mode='r', newline='') as file:
    reader = csv.DictReader(file)

    for row in reader:
        if row['department'] == 'IT':
          print(row)

{'emp_id': '1', 'name': 'Rahul', 'department': 'IT', 'salary': '70000', 'city': 'Hyderabad'}
{'emp_id': '3', 'name': 'Arjun', 'department': 'IT', 'salary': '75000', 'city': 'Chennai'}
{'emp_id': '5', 'name': 'Karan', 'department': 'IT', 'salary': '50000', 'city': 'Mumbai'}
{'emp_id': '8', 'name': 'Ravi', 'department': 'IT', 'salary': '72000', 'city': 'Hyderabad'}
{'emp_id': '11', 'name': 'Anita', 'department': 'IT', 'salary': '65000', 'city': 'Bangalore'}
{'emp_id': '13', 'name': 'Divya', 'department': 'IT', 'salary': '77000', 'city': 'Hyderabad'}
{'emp_id': '15', 'name': 'Pooja', 'department': 'IT', 'salary': '69000', 'city': 'Bangalore'}
{'emp_id': '18', 'name': 'Deepak', 'department': 'IT', 'salary': '73000', 'city': 'Hyderabad'}


4. Find employees with salary greater than 75,000.


In [68]:
high_earners = employees.filter(col("salary") > 75000)

high_earners.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



5. Calculate average salary.


In [69]:
employees.agg({"salary": "avg"}).show()

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+



6. Find highest paid employee.


In [70]:
employees.orderBy(col("salary").desc()).first()

Row(emp_id=20, name='Akash', department='Finance', salary=91000, city='Delhi')

7. Find lowest paid employee.


In [71]:
employees.orderBy(col("salary").asc()).first()

Row(emp_id=5, name='Karan', department='IT', salary=50000, city='Mumbai')

8. Count employees per department.


In [72]:
employees.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



9. Calculate average salary per department.


In [73]:
employees.groupBy("department").agg({"salary": "avg"}).show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



10. Find how many employees are in each city.

In [74]:
employees.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+




11. Find the top 5 highest salaries.

In [75]:
employees.orderBy(col("salary").desc()).limit(5).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+




12. Find employees working in Hyderabad with salary > 70k.

In [76]:
employees.filter((col("city") == "Hyderabad") & (col("salary") > 70000)).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



Dataset 3 — CSV (Sales Dataset)
File: sales.csv

Questions (Sales Analysis)
1. Calculate revenue per sale.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()


2. Calculate total revenue.


In [92]:

df = spark.read.csv('sales.csv', header=True, inferSchema=True)



3. Find revenue per product.


In [ ]:
df_with_revenue = df.withColumn("revenue", col("quantity") * col("price"))
df_with_revenue.show()


4. Find total quantity sold per product.

In [ ]:

total_revenue = df_with_revenue.agg(sum("revenue")).collect()[0][0]
print(f"Total Revenue: {total_revenue}")


5. Find best selling product.


In [ ]:

revenue_per_product = df_with_revenue.groupby("product").agg(sum("revenue").alias("total_revenue"))
revenue_per_product.show()


6. Find employee generating highest revenue.


In [ ]:

qty_per_product = df_with_revenue.groupby("product").agg(sum("quantity").alias("total_qty"))
qty_per_product.show()


7. Find average sale value.


In [ ]:

best_selling = qty_per_product.orderBy(desc("total_qty")).limit(1)
best_selling.show()


8. Find products generating revenue above 100,000.

In [ ]:

top_employee = df_with_revenue.groupby("emp_id").agg(sum("revenue").alias("emp_revenue")) \
    .orderBy(desc("emp_revenue")).limit(1)
top_employee.show()


Questions (JSON Processing)
1. Load the JSON file.


In [80]:
from google.colab import files
uploaded = files.upload()

Saving orders.json to orders.json


In [81]:
import json
from collections import defaultdict

with open('orders.json', 'r') as file:
    data = json.load(file)

orders = data['orders']



2. Print all orders.


In [82]:
for order in orders:
    print(order)


{'order_id': 1, 'customer': 'Rahul', 'product': 'Laptop', 'amount': 75000, 'city': 'Hyderabad'}
{'order_id': 2, 'customer': 'Sneha', 'product': 'Mouse', 'amount': 1500, 'city': 'Bangalore'}
{'order_id': 3, 'customer': 'Arjun', 'product': 'Keyboard', 'amount': 3000, 'city': 'Chennai'}
{'order_id': 4, 'customer': 'Priya', 'product': 'Laptop', 'amount': 75000, 'city': 'Hyderabad'}
{'order_id': 5, 'customer': 'Karan', 'product': 'Monitor', 'amount': 12000, 'city': 'Mumbai'}
{'order_id': 6, 'customer': 'Rahul', 'product': 'Mouse', 'amount': 1000, 'city': 'Hyderabad'}
{'order_id': 7, 'customer': 'Sneha', 'product': 'Laptop', 'amount': 75000, 'city': 'Bangalore'}
{'order_id': 8, 'customer': 'Arjun', 'product': 'Keyboard', 'amount': 3000, 'city': 'Chennai'}
{'order_id': 9, 'customer': 'Priya', 'product': 'Mouse', 'amount': 2000, 'city': 'Hyderabad'}
{'order_id': 10, 'customer': 'Rahul', 'product': 'Monitor', 'amount': 12000, 'city': 'Hyderabad'}


3. Count total orders.


In [83]:
total_orders = len(orders)
print(f"\nTotal Orders: {total_orders}")



Total Orders: 10


4. Calculate total sales amount.


In [84]:

total_sales = sum(order['amount'] for order in orders)
print(f"Total Sales Amount: {total_sales}")


Total Sales Amount: 259500


5. Find total spending per customer.


In [85]:

customer_spending = defaultdict(int)
for order in orders:
    customer_spending[order['customer']] += order['amount']

print("\nTotal Spending per Customer:")
for customer, amount in customer_spending.items():
    print(f"{customer}: {amount}")



Total Spending per Customer:
Rahul: 88000
Sneha: 76500
Arjun: 6000
Priya: 77000
Karan: 12000


6. Find highest spending customer.


In [86]:

highest_spender = max(customer_spending, key=customer_spending.get)
print(f"\nHighest Spending Customer: {highest_spender} ({customer_spending[highest_spender]})")



Highest Spending Customer: Rahul (88000)


7. Find total sales per product.


In [87]:

product_sales = defaultdict(int)
for order in orders:
    product_sales[order['product']] += order['amount']

print("\nTotal Sales per Product:")
for product, amount in product_sales.items():
    print(f"{product}: {amount}")



Total Sales per Product:
Laptop: 225000
Mouse: 4500
Keyboard: 6000
Monitor: 24000


8. Find customers from Hyderabad.


In [88]:
hyderabad_customers = list(set(order['customer'] for order in orders if order['city'] == 'Hyderabad'))
print(f"\nCustomers from Hyderabad: {hyderabad_customers}")


Customers from Hyderabad: ['Priya', 'Rahul']


9. Find orders with amount greater than 10,000.

In [89]:
high_value_orders = [order for order in orders if order['amount'] > 10000]
print("\nOrders > 10,000:")
for order in high_value_orders:
    print(order)




Orders > 10,000:
{'order_id': 1, 'customer': 'Rahul', 'product': 'Laptop', 'amount': 75000, 'city': 'Hyderabad'}
{'order_id': 4, 'customer': 'Priya', 'product': 'Laptop', 'amount': 75000, 'city': 'Hyderabad'}
{'order_id': 5, 'customer': 'Karan', 'product': 'Monitor', 'amount': 12000, 'city': 'Mumbai'}
{'order_id': 7, 'customer': 'Sneha', 'product': 'Laptop', 'amount': 75000, 'city': 'Bangalore'}
{'order_id': 10, 'customer': 'Rahul', 'product': 'Monitor', 'amount': 12000, 'city': 'Hyderabad'}



10. Count how many orders were placed in each city.

In [90]:
city_counts = defaultdict(int)
for order in orders:
    city_counts[order['city']] += 1

print("\nOrders per City:")
for city, count in city_counts.items():
    print(f"{city}: {count}")


Orders per City:
Hyderabad: 5
Bangalore: 2
Chennai: 2
Mumbai: 1


Final Combined Challenge
Using all datasets:
Tasks
1. Join employees.csv and sales.csv using emp_id .
2. Find total revenue generated by each employee.
3. Find top 5 employees by sales.
4. Find department generating highest revenue.
5. Save final result to:
final_sales_report.csv
Example output:
Employee Revenue
Rahul → 162000
Priya → 75000

In [93]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, desc

# TASK 1: Join employees and sales on emp_id
combined_df = employees.join(df, on="emp_id", how="inner")

# TASK 2: Find total revenue generated by each employee
combined_df = combined_df.withColumn("revenue", col("quantity") * col("price"))
employee_revenue = combined_df.groupby("name").agg(sum("revenue").alias("total_revenue"))

print("Employee Revenue:")
employee_revenue.show()

# TASK 3: Find top 5 employees by sales (revenue)
top_5_employees = employee_revenue.orderBy(desc("total_revenue")).limit(5)
print("Top 5 Employees:")
top_5_employees.show()

# TASK 4: Find department generating highest revenue
dept_revenue = combined_df.groupby("department").agg(sum("revenue").alias("dept_total_revenue")) \
    .orderBy(desc("dept_total_revenue"))

top_dept = dept_revenue.limit(1)
print("Top Department:")
top_dept.show()

# TASK 5: Save final result to final_sales_report.csv
# coalesce(1) ensures the output is a single CSV file
employee_revenue.coalesce(1).write.csv("final_sales_report.csv", header=True, mode="overwrite")

Employee Revenue:
+------+-------------+
|  name|total_revenue|
+------+-------------+
| Kunal|         1500|
| Sonal|        75000|
| Divya|        75000|
|  Ravi|         3000|
|Sanjay|         1500|
| Meera|        24000|
| Sneha|         1500|
| Priya|        75000|
|Vikram|        75000|
| Rahul|       162000|
| Anita|        12000|
| Manoj|         3000|
| Pooja|        12000|
| Arjun|         4000|
|  Amit|         2000|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+

Top 5 Employees:
+------+-------------+
|  name|total_revenue|
+------+-------------+
| Rahul|       162000|
| Sonal|        75000|
| Divya|        75000|
| Priya|        75000|
|Vikram|        75000|
+------+-------------+

Top Department:
+----------+------------------+
|department|dept_total_revenue|
+----------+------------------+
|        IT|            269500|
+----------+------------------+

